# Residual Stream Convergence

Whether representations converge across layers: layer-to-layer changes,
final stability, position convergence, norm growth, and summary.

## Why This Matters

Residual stream stream convergence characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_convergence import (
    layer_to_layer_convergence, final_representation_stability,
    position_convergence, norm_convergence,
    convergence_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer-to-Layer Convergence

Are updates getting smaller (converging)?

In [ ]:
result = layer_to_layer_convergence(model, tokens)
print(f"Converging: {result['is_converging']}, ratio: {result['convergence_ratio']:.4f}\n")
for c in result['per_layer']:
    bar = '█' * int(c['relative_change'] * 20)
    print(f"  Layer {c['layer']}: abs={c['absolute_change']:.4f}, "
          f"rel={c['relative_change']:.4f} {bar}")

## Final Representation Stability

When does the representation stabilize (become similar to final)?

In [ ]:
result = final_representation_stability(model, tokens)
print(f"Stabilization layer: {result['stabilization_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['cosine_to_final'] * 20)
    print(f"  Layer {p['layer']}: cos_to_final={p['cosine_to_final']:.4f} {bar}")

## Position Convergence

Do different positions converge to similar representations?

In [ ]:
result = position_convergence(model, tokens)
print(f"Position trend: {result['position_trend']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_pairwise_similarity'] * 20)
    print(f"  Layer {p['layer']}: pair_sim={p['mean_pairwise_similarity']:.4f} {bar}")

## Norm Convergence

Is the residual norm bounded or growing explosively?

In [ ]:
result = norm_convergence(model, tokens)
print(f"Growth factor: {result['norm_growth_factor']:.4f}, stable: {result['is_stable']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_norm'] * 5)
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f} ± {p['std_norm']:.4f} {bar}")

## Convergence Summary

In [ ]:
result = convergence_summary(model, tokens)
print(f"Converging: {result['is_converging']}, stable: {result['is_stable']}")
print(f"Stabilization layer: {result['stabilization_layer']}")
print(f"Norm growth: {result['norm_growth']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: change={p['relative_change']:.4f}, "
          f"cos_final={p['cosine_to_final']:.4f}, norm={p['mean_norm']:.4f}")